# Chapter 2: Searching State Spaces

```{admonition} Learning Objectives
:class: tip
- Understand state space representation as graphs
- Master uninformed search algorithms: BFS, DFS, UCS, Iterative Deepening
- Master informed search algorithms: Greedy Best-First, A*
- Apply local search methods: Hill Climbing, Simulated Annealing, Genetic Algorithms
- Solve Constraint Satisfaction Problems with various techniques
- Analyze algorithm complexity, optimality, and completeness
- Implement and compare different search strategies on real problems
```

```{epigraph}
The formulation of a problem is often more essential than its solution.

-- Albert Einstein
```

## 2.1 Introduction

Search is one of the most fundamental problem-solving techniques in artificial intelligence. Many AI problems can be formulated as finding a sequence of actions that transforms an initial state into a goal state. This paradigm applies to diverse applications:

- **Route Planning**: Finding optimal paths in maps (GPS navigation)
- **Puzzle Solving**: 8-puzzle, Rubik's cube, Sudoku
- **Game Playing**: Chess, checkers, Go (searching game trees)
- **Scheduling**: Resource allocation, task planning
- **Robot Motion Planning**: Navigation in physical space
- **Theorem Proving**: Finding proofs in logic

### Why Search Matters

Search is crucial because:
1. Many problems have no known direct solution formula
2. The solution space is too large to enumerate
3. We need systematic methods to explore possibilities
4. Different search strategies have different trade-offs (time, memory, optimality)

### Search vs. Planning

- **Search**: Finding a path through a state space (focuses on *how* to reach goal)
- **Planning**: Constructing a sequence of actions (focuses on *what* actions to take)

In practice, the distinction blurs - both involve exploring state spaces to find solutions.

### 2.1.1 State Space as a Graph

The **state space** of a problem is modeled as a directed graph:

**Components:**

1. **States (Nodes)**: Represent configurations of the world
   - Example (8-puzzle): Arrangement of tiles
   - Example (route planning): Geographic locations

2. **Actions (Edges)**: Transitions between states
   - Example (8-puzzle): Slide tile up/down/left/right
   - Example (route planning): Drive from city A to city B

3. **Initial State**: Starting configuration
   - Where the agent begins

4. **Goal State(s)**: Desired configuration(s)
   - What we're trying to achieve
   - Can be explicit state or test function

5. **Path Cost**: Sum of action costs in a sequence
   - Often uniform (1 per action) or weighted (different costs)

**State Space Graph Properties:**

- **Branching Factor (b)**: Average number of successors per state
- **Depth (d)**: Distance from initial state to shallowest goal
- **Maximum Depth (m)**: Longest path in state space
- **Size**: Total number of reachable states

These properties determine algorithm performance.

### Search Tree vs. State Space Graph

**State Space Graph:**
- Each state appears once
- Captures actual problem structure
- May contain cycles

**Search Tree:**
- States can repeat on different paths
- Root is initial state
- Represents exploration process
- Can be infinite even if state space is finite (due to cycles)

Most search algorithms work on the search tree but avoid revisiting states using a visited/explored set.

In [ ]:
# Essential imports
from collections import deque, defaultdict
from typing import List, Set, Optional, Callable, Tuple, Any, Dict
import heapq
import random
import math
import time
from abc import ABC, abstractmethod

# For visualization (optional)
try:
    import matplotlib.pyplot as plt
    import numpy as np
    HAS_MATPLOTLIB = True
except ImportError:
    HAS_MATPLOTLIB = False
    print("Matplotlib not available - visualizations disabled")

print('✓ Libraries imported successfully')

## 2.2 Uninformed Search Algorithms

**Uninformed** (or **blind**) search strategies use only information available in the problem definition. They don't use problem-specific knowledge to guide the search.

Key uninformed strategies:
1. Breadth-First Search (BFS)
2. Depth-First Search (DFS)
3. Uniform-Cost Search (UCS)
4. Depth-Limited Search
5. Iterative Deepening Depth-First Search (IDDFS)
6. Bidirectional Search

### Evaluation Criteria

We evaluate search algorithms on:

1. **Completeness**: Does it always find a solution if one exists?
2. **Optimality**: Does it find the least-cost solution?
3. **Time Complexity**: How many nodes are generated/expanded?
4. **Space Complexity**: Maximum nodes stored in memory?

Complexity is typically expressed using:
- **b**: branching factor (max successors)
- **d**: depth of shallowest goal
- **m**: maximum depth of search tree

In [ ]:
class SearchProblem(ABC):
    """
    Abstract base class for search problems.
    Subclasses must implement get_actions, get_successor, and is_goal.
    """
    
    def __init__(self, initial_state, goal_state=None):
        self.initial_state = initial_state
        self.goal_state = goal_state
    
    @abstractmethod
    def is_goal(self, state) -> bool:
        """Test if state satisfies goal condition."""
        pass
    
    @abstractmethod
    def get_actions(self, state) -> List[Any]:
        """Return list of valid actions from this state."""
        pass
    
    @abstractmethod
    def get_successor(self, state, action) -> Any:
        """Return the state resulting from applying action to state."""
        pass
    
    def action_cost(self, state, action, next_state) -> float:
        """Return cost of applying action. Default is uniform cost of 1."""
        return 1.0
    
    def get_successors(self, state) -> List[Tuple[Any, Any, float]]:
        """
        Return list of (action, next_state, cost) tuples.
        Useful for some search algorithms.
        """
        successors = []
        for action in self.get_actions(state):
            next_state = self.get_successor(state, action)
            cost = self.action_cost(state, action, next_state)
            successors.append((action, next_state, cost))
        return successors


class SearchNode:
    """
    Node in the search tree.
    Contains state, parent reference, action taken, path cost, and depth.
    """
    
    def __init__(self, state, parent=None, action=None, path_cost=0.0, depth=0):
        self.state = state
        self.parent = parent
        self.action = action
        self.path_cost = path_cost
        self.depth = depth
    
    def expand(self, problem):
        """Generate child nodes for all possible actions."""
        children = []
        for action, next_state, cost in problem.get_successors(self.state):
            child = SearchNode(
                state=next_state,
                parent=self,
                action=action,
                path_cost=self.path_cost + cost,
                depth=self.depth + 1
            )
            children.append(child)
        return children
    
    def get_path(self):
        """Extract path from root to this node."""
        path = []
        node = self
        while node:
            path.append(node.state)
            node = node.parent
        return list(reversed(path))
    
    def get_actions(self):
        """Extract action sequence from root to this node."""
        actions = []
        node = self
        while node.parent:
            actions.append(node.action)
            node = node.parent
        return list(reversed(actions))
    
    def __lt__(self, other):
        """For priority queue ordering."""
        return self.path_cost < other.path_cost
    
    def __repr__(self):
        return f"Node(state={self.state}, depth={self.depth}, cost={self.path_cost:.2f})"


print('✓ Search framework classes defined')

### 2.2.1 Breadth-First Search (BFS)

BFS explores the shallowest nodes first, expanding all nodes at depth *d* before any nodes at depth *d+1*.

**Algorithm:**
1. Start with initial state in a FIFO queue
2. Repeatedly: remove front of queue, check if goal, add successors to back
3. Use visited set to avoid revisiting states

**Properties:**
- **Complete**: Yes (if b is finite)
- **Optimal**: Yes (if all action costs are equal)
- **Time**: O(b^d) - exponential in depth
- **Space**: O(b^d) - must store all nodes at current level

**Advantages:**
- Finds shallowest solution
- Simple to implement
- Guaranteed to find solution if one exists

**Disadvantages:**
- Memory requirements can be prohibitive
- Explores many irrelevant states
- Slow for deep solutions

**When to Use:**
- Solution is shallow
- State space is small
- Optimality (shortest path) is required
- All actions have same cost

In [ ]:
def breadth_first_search(problem: SearchProblem, verbose: bool = True) -> Optional[List]:
    """
    Breadth-First Search implementation.
    
    Args:
        problem: SearchProblem instance
        verbose: Print statistics if True
    
    Returns:
        List of states from initial to goal, or None if no solution
    """
    start_time = time.time()
    
    # Initialize frontier with initial state
    initial_node = SearchNode(problem.initial_state)
    frontier = deque([initial_node])  # FIFO queue
    
    # Track visited states to avoid cycles
    visited = {problem.initial_state}
    
    # Statistics
    nodes_expanded = 0
    max_frontier_size = 1
    
    while frontier:
        # Track maximum frontier size (space complexity indicator)
        max_frontier_size = max(max_frontier_size, len(frontier))
        
        # Get shallowest node
        node = frontier.popleft()
        nodes_expanded += 1
        
        # Goal test
        if problem.is_goal(node.state):
            elapsed = time.time() - start_time
            if verbose:
                print(f"✓ BFS: Solution found!")
                print(f"  Path length: {node.depth}")
                print(f"  Path cost: {node.path_cost:.2f}")
                print(f"  Nodes expanded: {nodes_expanded}")
                print(f"  Max frontier size: {max_frontier_size}")
                print(f"  Time: {elapsed:.4f}s")
            return node.get_path()
        
        # Expand node - add all successors to frontier
        for child in node.expand(problem):
            if child.state not in visited:
                visited.add(child.state)
                frontier.append(child)
    
    # No solution found
    if verbose:
        print("✗ BFS: No solution exists")
    return None


print('✓ BFS implemented')

### 2.2.2 A* Search

A* is the most widely used informed search algorithm. It combines the actual cost to reach a node with the estimated cost to the goal.

**Evaluation Function**: 
$$f(n) = g(n) + h(n)$$

Where:
- **g(n)**: Actual cost from start to node n
- **h(n)**: Estimated cost from n to goal (heuristic)
- **f(n)**: Estimated total cost of cheapest solution through n

**How It Works:**
1. Maintain frontier ordered by f(n) = g(n) + h(n)
2. Expand node with lowest f(n)
3. Goal test when node is expanded

**Properties** (with admissible heuristic):**
- **Complete**: Yes
- **Optimal**: Yes (guaranteed least-cost solution!)
- **Time**: Exponential, but can be much better than uninformed search
- **Space**: Keeps all nodes in memory (can be limiting)

**Why A* is Optimal:**

If h(n) is admissible:
1. A* expands all nodes with f(n) < C* (C* = optimal cost)
2. Never expands nodes with f(n) > C*
3. Therefore finds optimal solution

**Proof Intuition**: If A* returned suboptimal solution, there would be an unexpanded node on optimal path with f ≤ C*, contradicting that A* finished.

**A* vs. UCS:**
- UCS is A* with h(n) = 0
- A* uses h(n) to focus search toward goal
- With perfect heuristic (h = h*), A* expands only nodes on optimal path!

**When to Use A*:**
- Have good admissible heuristic
- Need optimal solution
- Can afford memory
- This is usually the best choice!

In [ ]:
def a_star_search(problem: SearchProblem,
                   heuristic: Callable[[Any], float],
                   verbose: bool = True) -> Optional[List]:
    """
    A* Search implementation.
    Guaranteed optimal if heuristic is admissible.
    
    Args:
        problem: SearchProblem instance
        heuristic: Admissible heuristic function h(n)
        verbose: Print statistics if True
    
    Returns:
        List of states from initial to goal, or None if no solution
    """
    start_time = time.time()
    
    # Priority queue: (f_value, counter, g_value, node)
    initial_node = SearchNode(problem.initial_state)
    h_init = heuristic(problem.initial_state)
    f_init = 0 + h_init  # g=0 initially
    frontier = [(f_init, 0, 0.0, initial_node)]
    counter = 1
    
    # Track best g-value found for each state
    best_g = {problem.initial_state: 0.0}
    
    nodes_expanded = 0
    max_frontier_size = 1
    
    while frontier:
        max_frontier_size = max(max_frontier_size, len(frontier))
        
        f_val, _, g_val, node = heapq.heappop(frontier)
        
        # Skip if we've found better path to this state
        if g_val > best_g.get(node.state, float('inf')):
            continue
        
        nodes_expanded += 1
        
        # Goal test on expansion (critical for optimality!)
        if problem.is_goal(node.state):
            elapsed = time.time() - start_time
            if verbose:
                print(f"✓ A*: Optimal solution found!")
                print(f"  Path length: {node.depth}")
                print(f"  Optimal cost: {node.path_cost:.2f}")
                print(f"  Nodes expanded: {nodes_expanded}")
                print(f"  Max frontier size: {max_frontier_size}")
                print(f"  Time: {elapsed:.4f}s")
            return node.get_path()
        
        # Expand node
        for child in node.expand(problem):
            # Only add if we found a better path
            if child.path_cost < best_g.get(child.state, float('inf')):
                best_g[child.state] = child.path_cost
                h_child = heuristic(child.state)
                f_child = child.path_cost + h_child
                heapq.heappush(frontier, (f_child, counter, child.path_cost, child))
                counter += 1
    
    if verbose:
        print("✗ A*: No solution exists")
    return None


print('✓ A* Search implemented')

## 2.3 Example: 8-Puzzle Problem

The 8-puzzle is a classic search problem. It consists of a 3×3 board with 8 numbered tiles and one blank space. The goal is to rearrange tiles from initial configuration to goal configuration.

**State Space:**
- States: All possible configurations of tiles (9!/2 = 181,440)
- Actions: Slide tile into blank space (up/down/left/right)
- Goal: Specific target configuration

**Heuristics for 8-Puzzle:**

1. **Misplaced Tiles**: Count how many tiles are not in goal position
   - Admissible (each misplaced tile needs at least 1 move)
   - Simple but not very informative

2. **Manhattan Distance**: Sum of distances each tile is from goal position
   - Admissible (can't move tile to goal in fewer moves)
   - Better than misplaced tiles (dominates it)
   - Much more informative

3. **Linear Conflict**: Manhattan distance + 2 × linear conflicts
   - Even better but more complex to compute

**Why Manhattan Distance is Admissible:**
Each tile must move at least its Manhattan distance to reach goal position. Since we can only move one tile per action, h ≤ h*.

In [ ]:
class EightPuzzle(SearchProblem):
    """
    8-Puzzle problem implementation.
    State is tuple of 9 integers (0 represents blank).
    """
    
    def __init__(self, initial, goal=None):
        if goal is None:
            # Standard goal: 1 2 3 4 5 6 7 8 0
            goal = tuple(range(1, 9)) + (0,)
        super().__init__(tuple(initial), tuple(goal))
    
    def find_blank(self, state):
        """Return index of blank (0) tile."""
        return state.index(0)
    
    def get_actions(self, state):
        """Return list of valid moves from this state."""
        blank = self.find_blank(state)
        row, col = blank // 3, blank % 3
        
        actions = []
        if row > 0: actions.append('UP')
        if row < 2: actions.append('DOWN')
        if col > 0: actions.append('LEFT')
        if col < 2: actions.append('RIGHT')
        
        return actions
    
    def get_successor(self, state, action):
        """Return state resulting from taking action."""
        blank = self.find_blank(state)
        row, col = blank // 3, blank % 3
        
        # Direction mappings
        moves = {
            'UP': (-1, 0),
            'DOWN': (1, 0),
            'LEFT': (0, -1),
            'RIGHT': (0, 1)
        }
        
        dr, dc = moves[action]
        new_row, new_col = row + dr, col + dc
        new_blank = new_row * 3 + new_col
        
        # Swap blank with adjacent tile
        state_list = list(state)
        state_list[blank], state_list[new_blank] = state_list[new_blank], state_list[blank]
        
        return tuple(state_list)
    
    def is_goal(self, state):
        return state == self.goal_state
    
    def display(self, state):
        """Pretty print the puzzle state."""
        for i in range(0, 9, 3):
            row = state[i:i+3]
            print(' '.join('_' if x == 0 else str(x) for x in row))
        print()


def manhattan_distance(state, goal=(1,2,3,4,5,6,7,8,0)):
    """
    Sum of Manhattan distances of each tile from goal position.
    Admissible and more informed than misplaced_tiles.
    """
    distance = 0
    for i in range(9):
        if state[i] != 0:  # Don't count blank
            # Find where this tile should be
            goal_idx = goal.index(state[i])
            
            # Current position
            curr_row, curr_col = i // 3, i % 3
            # Goal position
            goal_row, goal_col = goal_idx // 3, goal_idx % 3
            
            # Manhattan distance
            distance += abs(curr_row - goal_row) + abs(curr_col - goal_col)
    
    return distance


print('✓ 8-Puzzle implementation complete')

In [ ]:
# Test 8-Puzzle with different algorithms
print("=== 8-Puzzle: Comparing Search Algorithms ==")
print()

# Easy puzzle (depth 4)
initial = (1, 2, 3, 4, 0, 5, 7, 8, 6)
goal = (1, 2, 3, 4, 5, 6, 7, 8, 0)

puzzle = EightPuzzle(initial, goal)

print("Initial State:")
puzzle.display(initial)

print("Goal State:")
puzzle.display(goal)

print("--- BFS (uninformed) ---")
solution_bfs = breadth_first_search(puzzle)
print()

print("--- A* with Manhattan Distance ---")
solution_a2 = a_star_search(puzzle, lambda s: manhattan_distance(s, goal))
print()

print("Observation: A* with good heuristic expands fewer nodes than BFS!")

## Programming Tasks

### Task 1: N-Queens Problem (Medium)

Implement and compare solutions to the N-Queens problem:
- Solve for N=8 using backtracking (DFS-based)
- Count all distinct solutions
- Measure performance for N=4,8,12

### Task 2: Maze Solver (Easy-Medium)

Create a maze solver:
- Generate random maze (grid with walls)
- Implement BFS and A* solvers
- Compare path lengths and nodes expanded
- Use Manhattan distance as heuristic for A*

### Task 3: Sudoku Solver (Medium)

Implement Sudoku solving:
- Backtracking with constraint propagation
- MRV (Minimum Remaining Values) heuristic
- Compare performance on different difficulty levels

### Task 4: Route Planning (Hard)

Implement route planning:
- Create graph representation of cities
- Implement A* with Euclidean distance
- Compare with Dijkstra (UCS)
- Measure nodes expanded and runtime

### Task 5: Sliding Puzzle Variants (Medium-Hard)

Extend puzzle solving:
- Implement 15-puzzle (4x4)
- Test multiple heuristics (Manhattan, Linear Conflict)
- Compare heuristic effectiveness
- Analyze when problems become intractable

## Summary

This chapter covered fundamental search algorithms for AI problem-solving:

### Uninformed Search

| Algorithm | Complete | Optimal | Time | Space | Best For |
|-----------|----------|---------|------|-------|----------|
| BFS | Yes | Yes* | O(b^d) | O(b^d) | Shallow solutions, uniform costs |
| DFS | No | No | O(b^m) | O(bm) | Deep solutions, limited memory |
| UCS | Yes | Yes | O(b^C*) | O(b^C*) | Weighted graphs |
| IDDFS | Yes | Yes* | O(b^d) | O(bd) | Unknown depth, limited memory |

*Optimal for uniform costs

### Informed Search

| Algorithm | Complete | Optimal | Notes |
|-----------|----------|---------|-------|
| Greedy | No | No | Fast but can be misled |
| A* | Yes | Yes | Best choice if admissible heuristic available |

**A* Optimality**: Guaranteed with admissible heuristic

### Key Takeaways

1. **Problem Formulation Matters**: Choice of state representation affects efficiency
2. **Heuristics Are Crucial**: Good heuristics dramatically reduce search
3. **Trade-offs Exist**: Time vs. space, optimality vs. speed
4. **Domain Knowledge Helps**: Informed search beats blind search
5. **No Universal Best**: Algorithm choice depends on problem characteristics

### Practical Guidelines

**Choose Systematic Search When:**
- Need to find a path (not just a goal state)
- Optimality is important
- State space is manageable

**Use A* When:**
- Have good admissible heuristic
- Can afford memory
- Need optimal solution
- This is the default choice for most problems!

In the next chapter, we'll explore adversarial search for game playing, where multiple agents compete.

## Further Reading

### Textbooks
- Aggarwal, C. C. (2021). *Artificial Intelligence: A Textbook*. Springer. [Chapter 2]
- Russell, S., & Norvig, P. (2020). *Artificial Intelligence: A Modern Approach* (4th ed.). [Chapters 3-4]
- Pearl, J. (1984). *Heuristics: Intelligent Search Strategies*. Addison-Wesley.

### Seminal Papers
- Hart, P. E., Nilsson, N. J., & Raphael, B. (1968). A Formal Basis for the Heuristic Determination of Minimum Cost Paths. *IEEE Transactions on Systems Science and Cybernetics*.
- Korf, R. E. (1985). Depth-first iterative-deepening: An optimal admissible tree search. *Artificial Intelligence*, 27(1), 97-109.

### Online Resources
- [Red Blob Games - Pathfinding](https://www.redblobgames.com/pathfinding/) - Interactive visualizations
- [Stanford CS221](https://stanford.edu/~shervine/teaching/cs-221/)
- [PathFinding.js Visual](https://qiao.github.io/PathFinding.js/visual/)